# **Ablation Study: Isolation Forest (MushroomB)**

## **Section 1: Setup**

In [1]:
%%capture
!pip install imagededup

In [2]:
import os
import json
import glob
import shutil
import pickle
import inspect
import zipfile
import importlib.metadata as importlib_metadata

import numpy as np
import pandas as pd
from imagededup.methods import CNN
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import normalize

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
drive_dir = '/content/drive/MyDrive/projectMushroom'
data_name = 'MushroomB'

orig_bad_data_dir = f'{drive_dir}/data/{data_name}/imagesQuarantine'   # ผลรอบแรกสุด
raw_zip_path = f'{drive_dir}/data/{data_name}/{data_name}_RAW.zip'
addition_data_dir = f'{drive_dir}/addition/data/{data_name}'           # ผลใหม่ทั้งหมด
repro_bad_data_dir = f'{addition_data_dir}/imagesQuarantine'
candidates_dir = f'{addition_data_dir}/candidates'
figures_dir = f'{addition_data_dir}/figures'

os.makedirs(repro_bad_data_dir, exist_ok=True)
os.chdir(drive_dir)
print(f'Current directory: {os.getcwd()}')

Current directory: /content/drive/MyDrive/projectMushroom


In [5]:
import src.reproducibility as reproducibility
import src.preprocessing as preprocessing
import src.visualization as visualization

visualization.matplotlib_font(font_dir=f'{drive_dir}/assets/fonts', prefer_name='Times-Roman')

SEED = 42
reproducibility.global_seed(seed=SEED)

Global Seed set to: 42


In [6]:
runtime_data_dir = '/content/data'
if os.path.exists(runtime_data_dir) and any(os.scandir(runtime_data_dir)):
    raise RuntimeError(f"[Error] {runtime_data_dir} is not empty. Use 'Disconnect and delete runtime' first.")

with zipfile.ZipFile(raw_zip_path, 'r') as zip_ref:
    zip_ref.extractall(runtime_data_dir)
print(f'Dataset has been extracted to {runtime_data_dir}')

Dataset has been extracted to /content/data


## **Section 2: Reproduce detection**

In [7]:
verify_report_df = pd.DataFrame(preprocessing.verify_images(data_dir=runtime_data_dir, check_solid=True))
verify_report_df.to_csv(f'{repro_bad_data_dir}/{data_name}_VERIFY.csv', index=False)

for path in verify_report_df.loc[verify_report_df['issue'] == 'corrupted', 'filepaths']:
    os.remove(path)
    print(f'Removed: {path}')

[Log]: Extensions detect .jpg
[Log]: Total images found: 3017


Verifying images: 100%|██████████| 3017/3017 [00:59<00:00, 50.99it/s] 



--- Summary ---
Total issues found: 0


In [8]:
outlier_report = preprocessing.find_outliers(data_dir=runtime_data_dir, seed=SEED, contamination=0.05)
pd.DataFrame(outlier_report).to_csv(f'{repro_bad_data_dir}/{data_name}_OUTLIER.csv', index=False)

hashes_dict = preprocessing.find_hasher(data_dir=runtime_data_dir, output_path=f'{repro_bad_data_dir}/{data_name}_HASHES.pkl')
intra_report, inter_report = preprocessing.find_duplicates(hashes_dict=hashes_dict)
pd.DataFrame(intra_report).to_csv(f'{repro_bad_data_dir}/{data_name}_INTRA_DUPLICATES.csv', index=False)
pd.DataFrame(inter_report).to_csv(f'{repro_bad_data_dir}/{data_name}_INTER_DUPLICATES.csv', index=False)

Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 109MB/s]
2026-11-08 14:16:59,765: INFO Device set to cpu ..
INFO:imagededup.methods.cnn:Device set to cpu ..
2026-11-08 14:16:59,773: INFO Initialized: mobilenet_v3_small for feature extraction ..
INFO:imagededup.methods.cnn:Initialized: mobilenet_v3_small for feature extraction ..
2026-11-08 14:16:59,780: INFO Start: Image encoding generation
INFO:imagededup.methods.cnn:Start: Image encoding generation


Found 16 folders containing images.


2026-11-08 14:17:07,325: INFO End: Image encoding generation
INFO:imagededup.methods.cnn:End: Image encoding generation
2026-11-08 14:17:08,018: INFO Start: Image encoding generation
INFO:imagededup.methods.cnn:Start: Image encoding generation


Class Russula_Delica-NotEdible: Found 8 candidate outlier.


2026-11-08 14:17:19,743: INFO End: Image encoding generation
INFO:imagededup.methods.cnn:End: Image encoding generation
2026-11-08 14:17:20,261: INFO Start: Image encoding generation
INFO:imagededup.methods.cnn:Start: Image encoding generation


... (31 outputs left out)

100%|██████████| 152/152 [00:06<00:00, 23.59it/s]
2026-11-08 14:19:11,269: INFO End: Calculating hashes!
INFO:imagededup.methods.hashing:End: Calculating hashes!
2026-11-08 14:19:11,276: INFO Start: Calculating hashes...
INFO:imagededup.methods.hashing:Start: Calculating hashes...
100%|██████████| 245/245 [00:06<00:00, 37.57it/s]
2026-11-08 14:19:17,892: INFO End: Calculating hashes!
INFO:imagededup.methods.hashing:End: Calculating hashes!
2026-11-08 14:19:17,900: INFO Start: Calculating hashes...
INFO:imagededup.methods.hashing:Start: Calculating hashes...
100%|██████████| 156/156 [00:06<00:00, 23.40it/s]
2026-11-08 14:19:24,656: INFO End: Calculating hashes!
INFO:imagededup.methods.hashing:End: Calculating hashes!
2026-11-08 14:19:24,662: INFO Start: Calculating hashes...
INFO:imagededup.methods.hashing:Start: Calculating hashes...
100%|██████████| 129/129 [00:01<00:00, 67.41it/s]
2026-11-08 14:19:26,643: INFO End: Calculating hashes!
INFO:imagededup.methods.hashing:End: Calculating 

Saved hashes for 16 unique classes to /content/drive/MyDrive/projectMushroom/addition/data/MushroomB/imagesQuarantine/MushroomB_HASHES.pkl


Processing Classes: 100%|██████████| 16/16 [00:00<00:00, 3532.05it/s]


Found 3017 images in total.


In [9]:
REPORTS = {
    'VERIFY': ['label', 'filename', 'issue'],
    'OUTLIER': ['label', 'filename', 'issue', 'score'],
    'INTRA_DUPLICATES': ['label', 'filename', 'issue', 'hash_group', 'keep'],
    'INTER_DUPLICATES': ['label', 'filename', 'issue', 'hash_group', 'conflicting_labels'],
}

def load_report(bad_dir, report, cols):
    df = pd.read_csv(f'{bad_dir}/{data_name}_{report}.csv', dtype={'hash_group': str})
    return df[cols].sort_values(['label', 'filename']).reset_index(drop=True)

def load_hashes(bad_dir):
    with open(f'{bad_dir}/{data_name}_HASHES.pkl', 'rb') as f:
        hashes = pickle.load(f)
    return {(label, os.path.basename(path)): h for label, files in hashes.items() for path, h in files.items()}

results = []
for report, cols in REPORTS.items():
    orig_df = load_report(orig_bad_data_dir, report, cols)
    repro_df = load_report(repro_bad_data_dir, report, cols)
    identical = orig_df.equals(repro_df)
    results.append({'report': report, 'original_rows': len(orig_df), 'repro_rows': len(repro_df), 'identical': identical})
    if not identical:
        display(orig_df.merge(repro_df, how='outer', indicator=True).query("_merge != 'both'"))

orig_hashes, repro_hashes = load_hashes(orig_bad_data_dir), load_hashes(repro_bad_data_dir)
results.append({'report': 'HASHES', 'original_rows': len(orig_hashes), 'repro_rows': len(repro_hashes),
                'identical': orig_hashes == repro_hashes})

repro_summary_df = pd.DataFrame(results)
repro_summary_df.to_csv(f'{addition_data_dir}/{data_name}_REPRO_COMPARISON.csv', index=False)
display(repro_summary_df)
assert repro_summary_df['identical'].all(), '[Error] Reproduction is not identical to the first run.'
print(f'{data_name}: IDENTICAL to the first run')

,report,original_rows,repro_rows,identical
0,VERIFY,0,0,True
1,OUTLIER,158,158,True
2,INTRA_DUPLICATES,36,36,True
3,INTER_DUPLICATES,0,0,True
4,HASHES,3017,3017,True


MushroomB: IDENTICAL to the first run


## **Section 3: Isolation Forest configurations**

In [10]:
EXTENSIONS = ('.jpg', '.jpeg', '.png')

cnn = CNN()
encodings_dict = {}
for root, dirs, files in os.walk(runtime_data_dir):
    if any(f.lower().endswith(EXTENSIONS) for f in files):
        encodings = cnn.encode_images(image_dir=root)
        encodings_dict[os.path.basename(root)] = {
            'dir': root,
            'filenames': list(encodings.keys()),
            'vectors': normalize(np.array(list(encodings.values())), norm='l2'),
        }
print(f'Encoded {sum(len(v["filenames"]) for v in encodings_dict.values())} images in {len(encodings_dict)} classes')

2026-11-08 14:20:28,788: INFO Device set to cpu ..
INFO:imagededup.methods.cnn:Device set to cpu ..
2026-11-08 14:20:28,796: INFO Initialized: mobilenet_v3_small for feature extraction ..
INFO:imagededup.methods.cnn:Initialized: mobilenet_v3_small for feature extraction ..
2026-11-08 14:20:28,800: INFO Start: Image encoding generation
INFO:imagededup.methods.cnn:Start: Image encoding generation
2026-11-08 14:20:36,318: INFO End: Image encoding generation
INFO:imagededup.methods.cnn:End: Image encoding generation
2026-11-08 14:20:36,327: INFO Start: Image encoding generation
INFO:imagededup.methods.cnn:Start: Image encoding generation
2026-11-08 14:20:47,047: INFO End: Image encoding generation
INFO:imagededup.methods.cnn:End: Image encoding generation
2026-11-08 14:20:47,058: INFO Start: Image encoding generation
INFO:imagededup.methods.cnn:Start: Image encoding generation
2026-11-08 14:20:55,154: INFO End: Image encoding generation
INFO:imagededup.methods.cnn:End: Image encoding gener

Encoded 3017 images in 16 classes


In [11]:
CONFIGS = [
    {'n_estimators': 100, 'contamination': 'auto'},     # default ของ scikit-learn
    {'n_estimators': 200, 'contamination': 'auto'},
    {'n_estimators': 200, 'contamination': 0.01},
    {'n_estimators': 200, 'contamination': 0.05},       # ค่าที่ใช้ในรอบแรก
    {'n_estimators': 200, 'contamination': 0.10},
]
BASE_CONFIG = 'n200_c0.05'                              # ค่าที่ใช้ในรอบแรก

def config_name(cfg):
    return f"n{cfg['n_estimators']}_c{cfg['contamination']}"

def make_forest(cfg):
    return IsolationForest(contamination=cfg['contamination'], n_estimators=cfg['n_estimators'],
                           random_state=SEED, n_jobs=-1)

rows = []
for cfg in CONFIGS:
    for label, enc in encodings_dict.items():
        clf = make_forest(cfg).fit(enc['vectors'])
        preds = clf.predict(enc['vectors'])
        scores = clf.decision_function(enc['vectors'])
        for filename, pred, score in zip(enc['filenames'], preds, scores):
            rows.append({'config': config_name(cfg), 'n_estimators': cfg['n_estimators'],
                         'contamination': str(cfg['contamination']), 'label': label,
                         'filename': filename, 'score': score, 'flagged': pred == -1})

scores_df = pd.DataFrame(rows)
scores_df.to_csv(f'{addition_data_dir}/{data_name}_IF_SCORES.csv', index=False)

flagged_sets = {c: set(zip(g.loc[g['flagged'], 'label'], g.loc[g['flagged'], 'filename']))
                for c, g in scores_df.groupby('config', sort=False)}
n_images = len(scores_df[scores_df['config'] == BASE_CONFIG])
base_flagged = flagged_sets[BASE_CONFIG]

cols = ['label', 'filename', 'score']
orig_outlier_df = pd.read_csv(f'{orig_bad_data_dir}/{data_name}_OUTLIER.csv', float_precision='round_trip')
base_df = scores_df[(scores_df['config'] == BASE_CONFIG) & scores_df['flagged']]
assert (orig_outlier_df[cols].sort_values(['label', 'filename']).reset_index(drop=True)
        .equals(base_df[cols].sort_values(['label', 'filename']).reset_index(drop=True))), \
    f'[Error] {BASE_CONFIG} does not match the first-run _OUTLIER.csv'

display(pd.DataFrame([{'config': c, 'flagged': len(f), 'flagged_%': round(100 * len(f) / n_images, 2),
                       'overlap_with_first_run': len(f & base_flagged)} for c, f in flagged_sets.items()]))

,config,flagged,flagged_%,overlap_with_first_run
0,n100_cauto,29,0.96,29
1,n200_cauto,31,1.03,31
2,n200_c0.01,40,1.33,40
3,n200_c0.05,158,5.24,158
4,n200_c0.1,307,10.18,158


In [12]:
spec = {
    'imagededup_version': importlib_metadata.version('imagededup'),
    'scikit_learn_version': importlib_metadata.version('scikit-learn'),
    'cnn_class': f'{CNN.__module__}.{CNN.__name__}',
    'cnn_attributes': {k: type(v).__name__ for k, v in vars(cnn).items()},
    'feature_dim': int(next(iter(encodings_dict.values()))['vectors'].shape[1]),
    'feature_transform': str(cnn.transform),
    'feature_normalization': 'L2 (sklearn.preprocessing.normalize)',
    'fit_scope': 'per class',
    'isolation_forest_params': {config_name(cfg): make_forest(cfg).get_params() for cfg in CONFIGS},
}
with open(f'{addition_data_dir}/{data_name}_IF_SPEC.json', 'w') as f:
    json.dump(spec, f, indent=2, default=str)
print(json.dumps(spec, indent=2, default=str))
print(inspect.getsource(CNN.__init__))
print(inspect.getsource(type(cnn.model)))
print(cnn.transform)

{
  "imagededup_version": "0.3.3.post2",
  "scikit_learn_version": "1.6.1",
  "cnn_class": "imagededup.methods.cnn.CNN",
  "cnn_attributes": {
    "model_config": "CustomModel",
    "logger": "Logger",
    "batch_size": "int",
    "device": "device",
    "model": "MobilenetV3",
    "transform": "Compose",
    "verbose": "int",
    "dataloader": "DataLoader",
    "encoding_map": "dict"
  },
  "feature_dim": 576,
  "feature_transform": "Compose(\n    Resize(size=(256, 256), interpolation=bilinear, max_size=None, antialias=True)\n    CenterCrop(size=(224, 224))\n    ToTensor()\n    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])\n)",
  "feature_normalization": "L2 (sklearn.preprocessing.normalize)",
  "fit_scope": "per class",
  "isolation_forest_params": {
    "n100_cauto": {
      "bootstrap": false,
      "contamination": "auto",
      "max_features": 1.0,
      "max_samples": "auto",
      "n_estimators": 100,
      "n_jobs": -1,
      "random_state": 42,
      "verbo

## **Section 4: Results**

### **Ablation results**
`removed_first_run` = ภาพที่ถูก flag และถูกลบเป็น outlier ในรอบแรก (`TO_DELETE.csv`) / `overlap_with_base` = ภาพที่ถูก flag และ config ของรอบแรก flag ด้วย / ไม่นับภาพที่ขั้น verify พบ

In [13]:
to_delete_df = pd.read_csv(f'{orig_bad_data_dir}/{data_name}_TO_DELETE.csv')
removed_df = to_delete_df[to_delete_df['issue'] == 'candidateOutlier']
removed_first_run = set(zip(removed_df['label'], removed_df['filename']))
assert removed_first_run <= base_flagged, f'[Error] Removed outliers of the first run are not all flagged by {BASE_CONFIG}'

results_df = pd.DataFrame([{
    'n_estimators': cfg['n_estimators'],
    'contamination': str(cfg['contamination']),
    'flagged': len(f),
    'flagged_%': round(100 * len(f) / n_images, 2),
    'overlap_with_base': len(f & base_flagged),
    'removed_first_run': len(f & removed_first_run),
} for cfg in CONFIGS for f in [flagged_sets[config_name(cfg)]]])

results_df.to_csv(f'{addition_data_dir}/{data_name}_ABLATION_RESULTS.csv', index=False)
print(f'Removed as outliers in the first run: {len(removed_first_run)} | Evaluated images: {n_images}')
display(results_df)

Removed as outliers in the first run: 0 | Evaluated images: 3017


,n_estimators,contamination,flagged,flagged_%,overlap_with_base,removed_first_run
0,100,auto,29,0.96,29,0
1,200,auto,31,1.03,31,0
2,200,0.01,40,1.33,40,0
3,200,0.05,158,5.24,158,0
4,200,0.1,307,10.18,158,0


### **Records and figures**
`candidates/{config}/` เก็บภาพที่แต่ละ config flag / figure แสดงภาพที่ถูกลบเป็น outlier ในรอบแรกทั้งหมด

In [14]:
def copy_image(label, filename, dest_path):
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    shutil.copy2(os.path.join(encodings_dict[label]['dir'], filename), dest_path)

def save_figures(keys, name):
    fig_df = pd.DataFrame(sorted(keys), columns=['label', 'filename'])
    fig_df['filepaths'] = [os.path.join(encodings_dict[l]['dir'], f) for l, f in zip(fig_df['label'], fig_df['filename'])]
    fig_df.to_csv(f'{figures_dir}/{data_name}_{name}.csv', index=False)
    if fig_df.empty:
        print(f'{data_name}: no images for {name}.')
    for i, start in enumerate(range(0, len(fig_df), 10), start=1):  # layout ของ vis_images: 2 x 5
        visualization.vis_images(df=fig_df.iloc[start:start + 10], filepaths_col='filepaths', label_col='label',
                                 n_rows=2, n_cols=5, title=None, save_path=f'{figures_dir}/{data_name}_{name}_{i}')

for c, f in flagged_sets.items():
    if not os.path.exists(f'{candidates_dir}/{c}'):
        for label, filename in sorted(f):
            copy_image(label, filename, f'{candidates_dir}/{c}/{label}/{filename}')

os.makedirs(figures_dir, exist_ok=True)
save_figures(removed_first_run, 'REMOVED_FIRST_RUN')

MushroomB: no images for REMOVED_FIRST_RUN.
